# C — BERT, RoBERTa và DistilBERT của lượt chạy cục bộ
Notebook kiểm bộ 9 checkpoint do `scripts.pipeline.complete_project` tạo trên máy này.
Mỗi kiến trúc có ba seed 42, 123, 2026. Kết quả được đối chiếu với `reports/project_results/`,
không dùng bảng số hoặc checkpoint riêng của báo cáo 2.1 thay cho lượt chạy mới.
Run All đọc hồ sơ, kiểm BCE và suy luận test thật; không huấn luyện lại hoặc tìm ngưỡng bằng test.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/notebook_support.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.notebook_support import artifacts_root
from src.datasets.goemotions import load_goemotions, multi_hot, sha256, REVISION
from src.evaluation.protocols import read_json
from src.evaluation.metrics import evaluate_multilabel
from src.models.transformer import METRIC_NAMES
ARTIFACTS = artifacts_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_colwidth', 100)
print('Python:', sys.version.split()[0])
print('Dataset revision:', REVISION)
print('Artifacts:', ARTIFACTS.name)

from src.notebook_support import replay_test, benchmark
from src.models.transformer import ARCHITECTURES
verified, frames, labels = benchmark(str(ARTIFACTS))
history = pd.DataFrame([dict(architecture=r['architecture'], seed=r['seed'], selected_epoch=r['selected_epoch'], **e)
                        for r in verified['histories'] for e in r['history']])
print('Đã kiểm 9 checkpoint và 72 dòng của lượt chạy cục bộ.')

## 1. Cấu hình thật, ba seed và checkpoint đã chọn

In [ ]:
configurations = []
for run in verified['histories']:
    configurations.append({**run['config'], 'architecture':run['architecture'], 'seed':run['seed'],
        'selected_epoch':run['selected_epoch'], 'model_revision':run['model_revision']})
display(pd.DataFrame(configurations)[['architecture','seed','learning_rate','epochs',
    'selected_epoch','batch_size','max_length','model_revision']])
print('Demo chọn bằng validation Macro-F1 @0.5:', verified['selection'])
assert len(configurations) == 9

## 2. Tokenize, head 28 logits và BCE trên batch train thật
    Cell nạp checkpoint seed 42 của mỗi kiến trúc; chỉ forward, không cập nhật weights.
    Kiểm loss model bằng BCEWithLogitsLoss, không dùng softmax/argmax ép một nhãn.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scripts.transformers.train_transformer import encoded_dataset, trim_padding_collate
torch.set_num_threads(4)
loss_checks = []
for architecture in ARCHITECTURES:
    folder = ARTIFACTS/f'data/processed/transformers/{architecture}/seed_42/full/standard/checkpoint'
    tokenizer = AutoTokenizer.from_pretrained(folder, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(folder, local_files_only=True).eval()
    dataset, _ = encoded_dataset(frames['train'].head(4), tokenizer, labels, 128)
    batch = trim_padding_collate([dataset[j] for j in range(4)])
    with torch.inference_mode():
        out = model(**batch)
        expected = torch.nn.functional.binary_cross_entropy_with_logits(out.logits, batch['labels'])
    torch.testing.assert_close(out.loss, expected)
    loss_checks.append(dict(architecture=architecture, input_shape=str(tuple(batch['input_ids'].shape)),
        target_shape=str(tuple(batch['labels'].shape)), logits_shape=str(tuple(out.logits.shape)),
        bce=float(out.loss), training_update=False))
    del model, dataset, batch, out
display(pd.DataFrame(loss_checks))

## 3. Code huấn luyện dùng chung cho ba kiến trúc
Cell hiển thị code; Run All không gọi huấn luyện. Checkpoint được chọn bằng validation.
Các lượt chạy cục bộ dùng cùng implementation, BCE 28 nhãn và protocol đã khóa.

In [ ]:
import inspect
from IPython.display import Code
from scripts.transformers import train_transformer
display(Code(inspect.getsource(train_transformer.main), language='python'))

## 4. Lịch sử loss và Macro-F1 validation của cả 9 run

In [ ]:
import matplotlib.pyplot as plt
display(history[['architecture','seed','epoch','train_loss','macro_f1','micro_f1','selected_epoch']])
fig, axes = plt.subplots(3, 2, figsize=(11, 9))
for row, architecture in enumerate(ARCHITECTURES):
    for seed, group in history[history.architecture == architecture].groupby('seed'):
        for col, metric in enumerate(('train_loss', 'macro_f1')):
            axes[row,col].plot(group.epoch, group[metric], marker='o', label=f'seed {seed}')
            axes[row,col].set(title=f'{architecture}: {metric}', xlabel='Epoch đã chạy')
            axes[row,col].legend()
plt.tight_layout(); plt.show(); plt.close()
print('Epoch được chọn ghi riêng theo từng run; chưa có sweep để kết luận siêu tham số tối ưu.')

## 5. Validation và lựa chọn mô hình trước test
    Macro-F1 fixed là tiêu chí chọn kiến trúc; tuned-validation dùng chính dữ liệu
    chọn ngưỡng nên không được coi là test độc lập.

In [ ]:
historical = pd.DataFrame(verified['records'])
display(historical[(historical.system.str.startswith('C_')) & (historical.split == 'validation')])
averages = pd.DataFrame(verified['averages'])
display(averages[(averages.system.str.startswith('C_')) & (averages.split == 'validation')]
        [['system','threshold_mode','n_runs','macro_f1_mean','macro_f1_std','micro_f1_mean','micro_f1_std']])
fresh_parts = []

## Suy luận test mới — bert, đủ ba seed
        Mỗi checkpoint chạy thực sự trên 5.427 câu; áp fixed/global/tuned đã khóa.
        Cell sẽ dừng nếu metric mới lệch bộ số báo cáo quá 1e-12.

In [ ]:
part = pd.concat([replay_test(ARTIFACTS, 'C_bert', seed)
                  for seed in (42, 123, 2026)], ignore_index=True)
fresh_parts.append(part)
display(part)
print('PASS bert: 3 checkpoint, 9 dòng test; sai lệch scores lớn nhất:', part.score_max_abs_diff.max())

## Suy luận test mới — roberta, đủ ba seed
        Mỗi checkpoint chạy thực sự trên 5.427 câu; áp fixed/global/tuned đã khóa.
        Cell sẽ dừng nếu metric mới lệch bộ số báo cáo quá 1e-12.

In [ ]:
part = pd.concat([replay_test(ARTIFACTS, 'C_roberta', seed)
                  for seed in (42, 123, 2026)], ignore_index=True)
fresh_parts.append(part)
display(part)
print('PASS roberta: 3 checkpoint, 9 dòng test; sai lệch scores lớn nhất:', part.score_max_abs_diff.max())

## Suy luận test mới — distilbert, đủ ba seed
        Mỗi checkpoint chạy thực sự trên 5.427 câu; áp fixed/global/tuned đã khóa.
        Cell sẽ dừng nếu metric mới lệch bộ số báo cáo quá 1e-12.

In [ ]:
part = pd.concat([replay_test(ARTIFACTS, 'C_distilbert', seed)
                  for seed in (42, 123, 2026)], ignore_index=True)
fresh_parts.append(part)
display(part)
print('PASS DistilBERT: 3 checkpoint, 9 dòng test; sai lệch scores lớn nhất:', part.score_max_abs_diff.max())

## 6. Tổng hợp test mean ± sample std từ kết quả suy luận mới

In [ ]:
from scripts.analysis.summarize_project import summarize
fresh = pd.concat(fresh_parts, ignore_index=True)
assert len(fresh) == 27 and (fresh.n_samples == 5427).all()
fresh_averages = summarize(fresh.to_dict('records'))
for row in fresh_averages:
    old = next(r for r in verified['averages'] if all(r[k] == row[k]
                for k in ('system','split','threshold_mode')))
    for metric in METRIC_NAMES:
        for suffix in ('_mean','_std'):
            row[metric+suffix+'_diff_vs_frozen'] = row[metric+suffix]-old[metric+suffix]
display(pd.DataFrame(fresh_averages)[['system','threshold_mode','n_runs',
    'macro_f1_mean','macro_f1_std','micro_f1_mean','micro_f1_std','hamming_loss_mean']])
print('Đã suy luận 27 dòng test và 9 nhóm. Chỉ số báo cáo tính chính xác từ scores gốc; replay mới có thể khác do AMP. Xem metric_max_abs_diff và changed_label_decisions, không ghi đè benchmark.')

## 7. Ví dụ dự đoán cùng ID từ scores vừa suy luận
    Ba mô hình dùng seed 123 và cùng câu test để đối chiếu. Mẫu lấy theo thứ tự dữ liệu,
    không sửa văn bản hay chọn lại ngưỡng theo kết quả test.

In [ ]:
from src.models.baseline import load_aligned_scores
examples = []
test = frames['test']
for architecture in ARCHITECTURES:
    scores = load_aligned_scores(ROOT/f'reports/notebook_replay/C_{architecture}_123_test.npz', test.id.tolist(), labels)
    protocol_path = ARTIFACTS/f'data/processed/transformers/{architecture}/seed_123/full/standard/final_protocol.json'
    protocol = read_json(protocol_path)
    thresholds = next(c['thresholds'] for c in protocol['configurations'] if c['threshold_mode']=='tuned')
    for i in range(3):
        examples.append(dict(architecture=architecture, id=test.id.iloc[i], text=test.text.iloc[i],
            truth=[labels[j] for j in test.labels.iloc[i]],
            predicted=[labels[j] for j in np.flatnonzero(scores[i] >= thresholds)]))
display(pd.DataFrame(examples))

## 8. Phạm vi kiểm chứng
Kiểm đủ 9 checkpoint của pipeline cục bộ, suy luận lại test và so metric với bảng mới.
Thời điểm huấn luyện và seed nằm trong metadata; thời điểm chạy notebook được ghi riêng.
Mô hình demo được chọn theo validation của lượt chạy này, không cố định kiến trúc thắng.

In [ ]:
display(pd.Series({'full_checkpoints_replayed': int(fresh.groupby(['system','seed']).ngroups),
    'test_rows_recomputed': len(fresh), 'test_groups_recomputed': len(fresh_averages),
    'max_score_difference': fresh.score_max_abs_diff.max(),
    'max_metric_difference': fresh.metric_max_abs_diff.max(), 'training_repeated': False}))